# Complexity 04c (C#) — k-server et work function : la conjecture tombée, mesurée en monde jouet

> **Bandeau preprint** : ce notebook s'appuie sur *The k-server conjecture is true*
> (Coester, Koutsoupias, Zbysiński, arXiv 2609.15979, soumis le 14/09/2026 — quelques jours
> avant l'écriture de ces lignes, **non relu**). Nous ne démontrons RIEN du théorème :
> nous implémentons l'algorithme (la work function algorithm, connue depuis 1990) en **C# BCL pur**
> et nous **mesurons les ratios réalisés** sur des instances jouets, pour vérifier que la garantie
> a le goût qu'elle promet. Même discipline d'honnêteté que Complexity-04b : mesurer, jamais re-prouver.

| | |
|---|---|
| **Module** | Décisions online sous incertitude — analyse compétitive |
| **Niveau** | Avancé (extension directe de Complexity-04b) |
| **Technologies** | C# / .NET Interactive — BCL pur (`System.Linq`, `System.Collections.Generic`) + figures SVG inline zéro-restauration |
| **Durée estimée** | 35-40 min — instances jouets CPU, exécution complète < 2 min |

**Objectifs d'apprentissage** — à la fin de ce notebook, vous saurez :
- [ ] formuler le problème k-server et le relier au caching (métrique uniforme) ;
- [ ] calculer l'optimum offline par programmation dynamique sur les configurations ;
- [ ] dérouler la work function algorithm et lire son critère `wf(A) + d(courant, A)` ;
- [ ] mesurer un ratio compétitif online/offline par instance, et distinguer
      « garantie pire cas ≤ k » (théorème) et « ratio réalisé ~1.1-1.5 » (mesure) ;
- [ ] construire une séquence qui piège LRU et montrer que WFA y résiste ;
- [ ] visualiser la work function comme une surface sur l'espace des configurations.

**Prérequis** : [Complexity-04b — conjectures online](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) (work function, métriques finies, conjectures de septembre 2026), et par transitivité [Complexity-04 — décider sans connaître la suite](Complexity-04-OnlineAlgorithms-CSharp.ipynb) — aucun prérequis mathématique au-delà d'une distance de métrique.

**Jumeau** : [Version Python](Complexity-04c-KServer-WorkFunction-Python.ipynb) — même algorithme, métriques et budgets, et la même **correction de l'Exercice 2** : dans le paging, LRU **est** k-compétitif (Sleator-Tarjan, 1985), et un ratio fini au-dessus de k — qui n'apparaît que quand LRU part d'un cache froid et OPT d'un cache chaud — ne réfute pas une garantie énoncée avec une **constante additive**.

**Navigation** : [Complexity-04b — conjectures online](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) ← · → [Complexity-04d — secrétaire matroidal](Complexity-04d-Secretaire-Matroidal-Python.ipynb) · [README série](README.md)

## 1. Le problème : servir des requêtes avec k serveurs, sans connaître la suite

Le fil de cette série est la **décision online sous incertitude** : on ne connaît pas la suite
et l'algorithme doit agir quand même. Deux voisins d'une autre série en montrent l'extrémité :
un bandit ([RL-04 — Bandits manchots](../../RL/RL-04-Bandits-Manchots-Python.ipynb)) ignore
quels bras paient, un agent Dyna-Q ([RL-08 — Dyna-Q, planification](../../RL/RL-08-Dyna-Q-Planification-Python.ipynb))
ignore la transition qu'il va déclencher.
Le problème **k-server** pousse cette ignorance à son cran d'arrêt : l'algorithme voit les
requêtes **une par une**, et doit **immédiatement** déplacer un serveur pour servir la
requête courante — sans jamais connaître la suivante.

> **Définition (k-server).** Un espace métrique à n points (une distance `d`), k serveurs
> placés sur k points distincts, une suite de requêtes `r_1, r_2, ...` (des points).
> À chaque requête `r_t`, si aucun serveur n'est sur `r_t`, l'algorithme choisit un serveur
> et le déplace ; il paie la distance parcourue. **Objectif : minimiser le coût total.**

La comparaison de référence est l'**optimum offline** (OPT) : le coût minimal d'un allocutaire
qui connaîtrait **toute la suite à l'avance**. Un algorithme online est **k-compétitif**
si son coût reste ≤ k × OPT sur toute suite — le facteur k est inévitable (borne inférieure :
aucun algorithme déterministe ne fait mieux que k sur toutes les métriques).

**Trente-six ans d'histoire en une phrase** : la conjecture (Manasse, McGeoch, Sleator, Tarjan,
1990) disait que la **work function algorithm** (WFA) atteint exactement ce ratio k sur tout
espace métrique ; Koutsoupias et Papadimitriou (1995) ont prouvé (2k−1)-compétitive — le
meilleur résultat connu pendant trente et un ans — et le preprint cité en tête clôt la conjecture :
WFA est k-compétitive, le rapport optimal.

Ce que ce notebook fait de concret :

1. implémenter WFA et OPT **from scratch** en C# BCL (aucune librairie d'algorithmique online) ;
2. vérifier **par instance** que le ratio WFA/OPT mesuré ne dépasse jamais k ;
3. opposer WFA à LRU/FIFO sur le paging — le cas particulier que tout développeur connaît ;
4. visualiser la work function, l'objet que la preuve manipule et que personne ne voit jamais.

In [1]:
// Imports BCL + culture invariant verrouillée : sorties numériques reproductibles,
// indépendantes de la locale du poste (point-virgule, exposants, séparateurs)
using System;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Text;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime .NET {Environment.Version}; culture {CultureInfo.CurrentCulture.Name}");
Console.WriteLine("Imports OK : System.Linq, System.Collections.Generic, System.Text — BCL pur, aucune librairie d'algorithmique online");

Runtime .NET 10.0.15; culture 


Imports OK : System.Linq, System.Collections.Generic, System.Text — BCL pur, aucune librairie d'algorithmique online


## 2. Trois métriques jouets

Tout vit dans une **fonction de distance** `d(i, j)`. Nous en utilisons trois, chacune
illustrant une facette du problème :

- la **ligne** : n points alignés, `d(i,j) = |i−j|` — le monde des serveurs mobiles ;
- le **cycle** : n points sur un anneau, distance au plus court des deux sens ;
- la **métrique uniforme** : `d(i,j) = 0` si même point, `1` sinon — c'est le **paging** :
  k serveurs = un cache de k cases, servir une requête = avoir la page en cache,
  un déplacement = un **miss** (coût 1).

In [2]:
// Ligne : n points 0..n-1, distance = valeur absolue de l'ecart.
Func<int,int,double> LigneMetric(int n) => (i,j) => (double)Math.Abs(i - j);
// Cycle : n points sur un anneau, plus court des deux chemins.
Func<int,int,double> CycleMetric(int n) => (i,j) => { int d = Math.Abs(i - j); return Math.Min(d, n - d); };
// Uniforme (paging) : 0 si meme point, 1 sinon.
Func<int,int,double> UnifMetric(int n) => (i,j) => (i == j ? 0.0 : 1.0);

var dLigne = LigneMetric(6);
var dCycle = CycleMetric(5);
var dUnif = UnifMetric(8);
Console.WriteLine($"Ligne  n=6 : d(0,5) = {dLigne(0, 5)}, d(2,4) = {dLigne(2, 4)}");
Console.WriteLine($"Cycle  n=5 : d(0,3) = {dCycle(0, 3)} (le ring raccourcit 3 -> 2)");
Console.WriteLine($"Unif.  n=8 : d(3,3) = {dUnif(3, 3)}, d(3,7) = {dUnif(3, 7)} (un miss de cache)");

Ligne  n=6 : d(0,5) = 5, d(2,4) = 2


Cycle  n=5 : d(0,3) = 2 (le ring raccourcit 3 -> 2)


Unif.  n=8 : d(3,3) = 0, d(3,7) = 1 (un miss de cache)


## 3. Configurations de serveurs

L'état complet du système à l'instant t est l'ensemble des positions des k serveurs —
une **configuration** : un sous-ensemble de taille k des n points. La distance entre deux
configurations est le coût minimal pour transformer l'une en l'autre (appariement optimal
des serveurs ; à k ≤ 4, l'énumération des k! appariements est immédiate).

In [3]:
// Prochaine permutation lexicographique de 0..k-1 : le BCL n'en a pas de brouillon.
// Renvoie false quand on a cycle sur toutes les permutations.
bool NextPermutation(int[] a) {
  int i = a.Length - 2;
  while (i >= 0 && a[i] > a[i + 1]) i--;
  if (i < 0) return false;
  int j = a.Length - 1;
  while (a[j] < a[i]) j--;
  (a[i], a[j]) = (a[j], a[i]);
  for (int l = i + 1, r = a.Length - 1; l < r; (l, r) = (l + 1, r - 1)) (a[l], a[r]) = (a[r], a[l]);
  return true;
}

// Toutes les configurations de k serveurs sur n points (combinaisons C(n,k)).
List<int[]> AllConfigs(int n, int k) {
  var res = new List<int[]>();
  var cur = new List<int>();
  void Rec(int start) {
    if (cur.Count == k) { res.Add(cur.ToArray()); return; }
    for (int i = start; i <= n - (k - cur.Count); i++) { cur.Add(i); Rec(i + 1); cur.RemoveAt(cur.Count - 1); }
  }
  Rec(0);
  return res;
}

// Cout minimal pour amener la config A vers la config B : appariement optimal,
// enumere les k! permutations de B (k <= 4).
double CfgDistance(int[] A, int[] B, Func<int,int,double> dist) {
  var idx = Enumerable.Range(0, B.Length).ToArray();
  double best = double.PositiveInfinity;
  do {
    double s = 0;
    for (int i = 0; i < A.Length; i++) s += dist(A[i], B[idx[i]]);
    if (s < best) best = s;
  } while (NextPermutation(idx));
  return best;
}

var cfgs42 = AllConfigs(4, 2);
var cfgLst = string.Join(", ", cfgs42.Select(c => "(" + string.Join(",", c) + ")"));
Console.WriteLine($"n=4, k=2 : {cfgs42.Count} configurations -> [{cfgLst}]");
Console.WriteLine($"d((0,1),(2,3)) sur la ligne = {CfgDistance(new[]{0,1}, new[]{2,3}, LigneMetric(4))} (les deux serveurs avancent de 2)");

n=4, k=2 : 6 configurations -> [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]


d((0,1),(2,3)) sur la ligne = 4 (les deux serveurs avancent de 2)


## 4. L'optimum offline est une programmation dynamique — et c'est la work function

Avant de jouer online, calculons OPT. Sur une suite `r_1..r_T`, le coût minimal pour
finalement finir dans une configuration `A` en ayant servi toutes les requêtes obéit à une
récurrence de DP classique :

```
wf_0(A) = d(config_initiale, A)
wf_t(A) = min sur B de [ wf_{t-1}(B) + d(B, A) ]     si r_t ∈ A
wf_t(A) = +infini                                    sinon  (A doit servir r_t)
```

Cette quantité a un nom : c'est la **work function**. Deux lectures, un seul objet :

- lecture offline : `min_A wf_T(A)` = **coût de l'optimum** OPT (la trajectoire minimale) ;
- lecture online : `wf_t(A)` dit combien coûterait le mieux-disant qui finirait en `A` —
  une **mémoire complète du passé**, recalculée à chaque requête.

Et l'algorithme ? La **work function algorithm** sert la requête `r_t` avec la configuration

```
A_t = argmin sur A ∋ r_t de [ wf_t(A) + d(A_{t-1}, A) ]
```

le meilleur compromis entre « finir pas trop cher selon le passé » (le terme wf) et « ne pas
trop bouger maintenant » (le terme d). Un seul moteur DP sert donc les deux rôles du notebook.

In [4]:
// Decoule WFA en ligne sur la suite de requetes.
// Retourne (cout_wfa, cout_opt, wf_final) :
//   cout_wfa : cout reellement paye par WFA (decisions online, pas de lookahead) ;
//   cout_opt : optimum offline = min_A wf_T(A), calcule par la MEME recurrence ;
//   wf_final : la work function a l'instant final (table config -> valeur).
(double wfa, double opt, Dictionary<int[],double> wf) WfaRun(
    int n, int k, Func<int,int,double> dist, int[] requests, int[] initCfg) {
  var C = AllConfigs(n, k);
  var init = (int[])initCfg.Clone(); Array.Sort(init);
  var wf = new Dictionary<int[],double>();
  foreach (var A in C) wf[A] = CfgDistance(init, A, dist);
  var cur = init;
  double total = 0.0;
  foreach (var r in requests) {
    var newWf = new Dictionary<int[],double>();
    foreach (var A in C) {
      if (!A.Contains(r)) { newWf[A] = double.PositiveInfinity; continue; }
      double best = double.PositiveInfinity;
      foreach (var B in C)
        if (wf[B] < double.PositiveInfinity) best = Math.Min(best, wf[B] + CfgDistance(B, A, dist));
      newWf[A] = best;
    }
    wf = newWf;
    var aStar = C.Where(A => A.Contains(r)).MinBy(A => wf[A] + CfgDistance(cur, A, dist));
    total += CfgDistance(cur, aStar, dist);
    cur = aStar;
  }
  double opt = C.Min(A => wf[A]);
  return (total, opt, wf);
}

// Exemple trace : ligne n=4, k=2, serveurs en (0,1), requetes [3,1,2,0,3]
var (w, o, _) = WfaRun(4, 2, LigneMetric(4), new int[]{3,1,2,0,3}, new int[]{0,1});
Console.WriteLine($"Exemple ligne n=4 k=2, requetes [3,1,2,0,3] : WFA = {w:0}, OPT = {o:0}, ratio = {w / o:0.00}");

Exemple ligne n=4 k=2, requetes [3,1,2,0,3] : WFA = 6, OPT = 6, ratio = 1,00


## 5. La garantie, goûtée instance par instance

Le théorème dit : **sur toute suite**, coût(WFA) ≤ k × OPT. Ce n'est pas une garantie en
moyenne — c'est par instance. Nous pouvons donc la tester telle quelle : générons des
centaines d'instances aléatoires (seedées, reproductibles), calculons les deux coûts,
et vérifions que le ratio ne franchit jamais k. Si votre implémentation est correcte,
la réponse est 0 violation sur tout l'échantillon.

**Honnêteté du banc** : ne pas confondre « aucune violation sur 600 instances » et « le
théorème est prouvé » — le pire cas théorique est une suite **adversariale construite**,
pas un tirage aléatoire (le ratio mesuré typique est ~1.1-1.6, loin du pire cas k ; c'est
précisément l'écart entre la garantie et la pratique que cette section rend visible).

In [5]:
// Retourne (nb_violations, ratios) : WFA > k*OPT compte comme violation.
(int violations, List<double> ratios) CheckCompetitivite(
    Func<int,Func<int,int,double>> metricFactory, int n = 6, int k = 3,
    int nInstances = 200, int T = 30, int seed = 7) {
  var rng = new Random(seed);
  var dist = metricFactory(n);
  var ratios = new List<double>();
  for (int i = 0; i < nInstances; i++) {
    var reqs = Enumerable.Range(0, T).Select(_ => rng.Next(n)).ToArray();
    var (ww, oo, _) = WfaRun(n, k, dist, reqs, Enumerable.Range(0, k).ToArray());
    if (oo > 0) ratios.Add(ww / oo);
  }
  int violations = ratios.Count(r => r > k + 1e-9);
  return (violations, ratios);
}

var banches = new (string nom, Func<int,Func<int,int,double>> fab)[] {
  ("ligne", LigneMetric), ("cycle", CycleMetric), ("uniforme", UnifMetric)
};
foreach (var (nom, fab) in banches) {
  var (viol, ratios) = CheckCompetitivite(fab, seed: 7);
  Console.WriteLine($"{nom,-9} n=6 k=3 : {ratios.Count} instances, ratio moy={ratios.Average():0.000}, max={ratios.Max():0.000}, violations de k=3 : {viol}");
}

ligne     n=6 k=3 : 200 instances, ratio moy=1,197, max=1,750, violations de k=3 : 0


cycle     n=6 k=3 : 200 instances, ratio moy=1,249, max=1,769, violations de k=3 : 0


uniforme  n=6 k=3 : 200 instances, ratio moy=1,558, max=2,286, violations de k=3 : 0


## 6. Mesures : ligne et cycle, adversaires faciles et difficiles

Deux familles de suites sur la ligne : des tirages **aléatoires** (l'adversaire indifferent)
et une **oscillation** entre les deux extrémités — le réflexe « follow the request » y paie
le prix fort, car il fait traverser la ligne à chaque requête. WFA, elle, absorbe
l'information et positionne ses serveurs.

Les figures sont des **SVG inline** émis en `text/html` par le kernel (technique zéro-restauration,
aucun paquet NuGet de charting) : elles rendent aussi bien en kernel live que sur GitHub ou hors-ligne.

In [6]:
using Microsoft.DotNet.Interactive.Formatting;
record PlotSvg(string Markup);
Formatter.Register(typeof(PlotSvg),
  (obj, writer) => ((TextWriter)writer).Write(((PlotSvg)obj).Markup), "text/html");

// Histogramme SVG : bins de valeurs + ligne verticale optionnelle (la garantie k).
// Coordonnees serialisees en culture invariante (une virgule decimal casserait la geometrie).
string SvgHistogram(IList<double> valeurs, int bins, double xCrit, string titre, string axeX) {
  var sb = new StringBuilder();
  int W = 560, H = 300, L = 46, B = 34, T = 30, R = 14;
  double pw = W - L - R, ph = H - T - B;
  double vmin = Math.Floor(valeurs.Min() * 10) / 10;
  double vmax = Math.Ceiling(valeurs.Max() * 10) / 10;
  if (vmax <= vmin) vmax = vmin + 1;
  double bw = (vmax - vmin) / bins;
  var counts = new int[bins];
  foreach (var v in valeurs) { int b = (int)Math.Min(bins - 1, Math.Floor((v - vmin) / bw)); counts[b]++; }
  int cmax = Math.Max(1, counts.Max());
  sb.Append($"<svg viewBox='0 0 {W} {H}' xmlns='http://www.w3.org/2000/svg' font-family='sans-serif'>");
  sb.Append($"<text x='{W / 2}' y='16' font-size='12' text-anchor='middle'>{titre}</text>");
  for (int i = 0; i < bins; i++) {
    if (counts[i] == 0) continue;
    double bh = (double)counts[i] / cmax * ph;
    double x = L + i * pw / bins;
    double y = T + ph - bh;
    sb.Append($"<rect x='{x.ToString("F1", CultureInfo.InvariantCulture)}' y='{y.ToString("F1", CultureInfo.InvariantCulture)}' width='{(pw / bins - 1).ToString("F1", CultureInfo.InvariantCulture)}' height='{bh.ToString("F1", CultureInfo.InvariantCulture)}' fill='#4C72B0'/>");
  }
  if (xCrit >= vmin && xCrit <= vmax) {
    double xx = L + (xCrit - vmin) / (vmax - vmin) * pw;
    sb.Append($"<line x1='{xx.ToString("F1", CultureInfo.InvariantCulture)}' y1='{T}' x2='{xx.ToString("F1", CultureInfo.InvariantCulture)}' y2='{T + ph}' stroke='crimson' stroke-dasharray='4 3' stroke-width='1.5'/>");
  }
  sb.Append($"<line x1='{L}' y1='{T + ph}' x2='{L + pw}' y2='{T + ph}' stroke='#333'/>");
  sb.Append($"<text x='{L + pw / 2}' y='{H - 8}' font-size='11' text-anchor='middle'>{axeX}  —  rouge pointille : garantie k</text>");
  sb.Append("</svg>");
  return sb.ToString();
}

var rng6 = new Random(20260916);
var ratiosLigne = new List<double>();
for (int i = 0; i < 300; i++) {
  var reqs = Enumerable.Range(0, 40).Select(_ => rng6.Next(6)).ToArray();
  var (ww, oo, _) = WfaRun(6, 2, LigneMetric(6), reqs, new int[]{0,1});
  if (oo > 0) ratiosLigne.Add(ww / oo);
}
var (wOsc, oOsc, _) = WfaRun(6, 2, LigneMetric(6),
  Enumerable.Range(0, 12).SelectMany(_ => new[]{5, 0}).ToArray(), new int[]{0,1});
var ratiosCycle = new List<double>();
for (int i = 0; i < 200; i++) {
  var reqs = Enumerable.Range(0, 30).Select(_ => rng6.Next(5)).ToArray();
  var (ww, oo, _) = WfaRun(5, 2, CycleMetric(5), reqs, new int[]{0,1});
  if (oo > 0) ratiosCycle.Add(ww / oo);
}
Console.WriteLine($"Ligne n=6 k=2 (300 suites) : ratio moy={ratiosLigne.Average():0.000}, max={ratiosLigne.Max():0.000}  [garantie theorique k=2]");
Console.WriteLine($"Ligne, oscillation 5/0 x12 : WFA={wOsc:0}, OPT={oOsc:0}, ratio={wOsc / oOsc:0.00}");
Console.WriteLine($"Cycle n=5 k=2 (200 suites) : ratio moy={ratiosCycle.Average():0.000}, max={ratiosCycle.Max():0.000}  [garantie theorique k=2]");
display(new PlotSvg(SvgHistogram(ratiosLigne, 24, 2.0, "Ligne n=6 k=2 : ratio WFA/OPT, 300 suites aleatoires", "ratio cout_WFA / cout_OPT")));
display(new PlotSvg(SvgHistogram(ratiosCycle, 20, 2.0, "Cycle n=5 k=2 : ratio WFA/OPT, 200 suites aleatoires", "ratio cout_WFA / cout_OPT")));

Ligne n=6 k=2 (300 suites) : ratio moy=1,117, max=1,483  [garantie theorique k=2]


Ligne, oscillation 5/0 x12 : WFA=4, OPT=4, ratio=1,00


Cycle n=5 k=2 (200 suites) : ratio moy=1,173, max=1,667  [garantie theorique k=2]


Ligne n=6 k=2 : ratio WFA/OPT, 300 suites aleatoires ratio cout_WFA / cout_OPT — rouge pointille : garantie k

Cycle n=5 k=2 : ratio WFA/OPT, 200 suites aleatoires ratio cout_WFA / cout_OPT — rouge pointille : garantie k

**Lecture** : les distributions vivent entre 1.0 et ~1.5, loin du plafond k=2 (ligne rouge
pointillée). La garantie du théorème est une **ceinture de sécurité pire-cas** ; la
conduite quotidienne est bien plus douce. L'oscillation 5/0 donne même un ratio de 1.0 :
WFA place un serveur à chaque extrémité puis ne bouge plus — l'adversaire naïf lui offre
la partie. Le dur adversaire de WFA existe (la preuve le construit), mais il ne se
rencontre pas par tirage au sort : c'est exactement la nuance que la section honnêteté
(§9) reformulera.

## 7. Paging : la métrique uniforme, et le duel WFA vs LRU vs FIFO

Sur la métrique uniforme, k-server **est** le caching : k serveurs = k cases de cache,
requêtes = pages demandées, coût = nombre de faults. Les heuristiques de remplacement
sont alors des algorithmes k-server à part entière :

- **LRU** (least recently used) : éjecte la page la plus anciennement utilisée ;
- **FIFO** : éjecte la plus anciennement entrée ;
- **OPT offline** : la politique de Belady (éjecter la page dont la prochaine référence
  est la plus lointaine) — que notre work function retrouve sans le savoir.

Le régime qui intéresse tout le monde : une **boucle cyclique** sur k+2 pages (5 pages pour k=3). LRU y
subit un fault à **chaque** requête (il éjecte toujours la page qui reviendra ensuite) —
le régime canonique où LRU frôle la borne sans la briser (60 < k × OPT = 96 ;
l'Exercice 2 construit la suite qui la dépasse). WFA, elle, doit rester ≤ k × OPT.

In [7]:
// LRU : ejecte la page la moins recemment utilisee. Retourne le nombre de faults.
double LruRun(int n, int k, int[] requests, int[] initPages) {
  var cache = new List<int>(initPages);
  int faults = 0;
  foreach (var r in requests) {
    if (cache.Contains(r)) { cache.Remove(r); cache.Add(r); }
    else { faults++; if (cache.Count == k) cache.RemoveAt(0); cache.Add(r); }
  }
  return faults;
}

// FIFO : ejecte la page entree en premier (pas de refresh au hit).
double FifoRun(int n, int k, int[] requests, int[] initPages) {
  var cache = new List<int>(initPages);
  var ordre = new Queue<int>(initPages);
  int faults = 0;
  foreach (var r in requests) {
    if (cache.Contains(r)) continue;
    faults++;
    if (cache.Count == k) { var sortie = ordre.Dequeue(); cache.Remove(sortie); }
    cache.Add(r); ordre.Enqueue(r);
  }
  return faults;
}

Console.WriteLine("LRU et FIFO definis : deux politiques de cache, deux algorithmes k-server");

LRU et FIFO definis : deux politiques de cache, deux algorithmes k-server


In [8]:
// Barres groupées SVG : 4 series (WFA/LRU/FIFO/OPT) par regime + ligne du plafond k x OPT.
string SvgGroupedBars(string[] labels, double[] wfa, double[] lru, double[] fifo, double[] opt,
                      double plafond, string titre) {
  var sb = new StringBuilder();
  int W = 780, H = 360, L = 48, B = 66, T = 34, R = 14;
  double pw = W - L - R, ph = H - T - B;
  double maxVal = Math.Max(plafond, new[] { wfa, lru, fifo, opt }.SelectMany(x => x).Max());
  if (maxVal <= 0) maxVal = 1;
  int g = labels.Length;
  double groupW = pw / g;
  var series = new (string nom, double[] v, string col)[] {
    ("WFA", wfa, "#4C72B0"), ("LRU", lru, "#DD8452"), ("FIFO", fifo, "#55A868"), ("OPT (offline)", opt, "#C44E52")
  };
  sb.Append($"<svg viewBox='0 0 {W} {H}' xmlns='http://www.w3.org/2000/svg' font-family='sans-serif'>");
  sb.Append($"<text x='{W / 2}' y='16' font-size='12' text-anchor='middle'>{titre}</text>");
  for (int i = 0; i < g; i++) {
    double gx = L + i * groupW;
    for (int s = 0; s < series.Length; s++) {
      double bh = series[s].v[i] / maxVal * ph;
      double x = gx + s * groupW / 4 + 1;
      double y = T + ph - bh;
      sb.Append($"<rect x='{x.ToString("F1", CultureInfo.InvariantCulture)}' y='{y.ToString("F1", CultureInfo.InvariantCulture)}' width='{(groupW / 4 - 2).ToString("F1", CultureInfo.InvariantCulture)}' height='{bh.ToString("F1", CultureInfo.InvariantCulture)}' fill='{series[s].col}'/>");
    }
    sb.Append($"<text x='{(gx + groupW / 2).ToString("F1", CultureInfo.InvariantCulture)}' y='{T + ph + 12}' font-size='9' text-anchor='middle'>{labels[i]}</text>");
  }
  double py = T + ph - plafond / maxVal * ph;
  sb.Append($"<line x1='{L}' y1='{py.ToString("F1", CultureInfo.InvariantCulture)}' x2='{L + pw}' y2='{py.ToString("F1", CultureInfo.InvariantCulture)}' stroke='dimgray' stroke-dasharray='5 3' stroke-width='1.2'/>");
  sb.Append($"<text x='{(L + pw).ToString("F1", CultureInfo.InvariantCulture)}' y='{(py - 4).ToString("F1", CultureInfo.InvariantCulture)}' font-size='9' text-anchor='end' fill='dimgray'>k x OPT (1re ligne)</text>");
  int lx = L;
  foreach (var sr in series) {
    sb.Append($"<rect x='{lx}' y='{H - 16}' width='10' height='10' fill='{sr.col}'/>");
    sb.Append($"<text x='{lx + 13}' y='{H - 7}' font-size='9'>{sr.nom}</text>");
    lx += 13 + sr.nom.Length * 5 + 16;
  }
  sb.Append("</svg>");
  return sb.ToString();
}

// Pour chaque regime (nom, requetes) : faults de WFA / LRU / FIFO / OPT.
(string[] noms, double[] wfa, double[] lru, double[] fifo, double[] opt) PagingTable(
    int n, int k, (string nom, int[] reqs)[] regimes, int[] initPages) {
  var dist = UnifMetric(n);
  var noms = new List<string>(); var wfa = new List<double>(); var lru = new List<double>();
  var fifo = new List<double>(); var opt = new List<double>();
  foreach (var (nom, reqs) in regimes) {
    var (w, o, _) = WfaRun(n, k, dist, reqs, initPages);
    noms.Add(nom); wfa.Add(w); lru.Add(LruRun(n, k, reqs, initPages));
    fifo.Add(FifoRun(n, k, reqs, initPages)); opt.Add(o);
  }
  return (noms.ToArray(), wfa.ToArray(), lru.ToArray(), fifo.ToArray(), opt.ToArray());
}

var rngP = new Random(20260916);
int NPAGES = 8, KCACHE = 3;
var initP = new int[]{0,1,2};
var regimes = new (string nom, int[] reqs)[] {
  ("boucle 3-4-5-6-7 (x12)", Enumerable.Range(0,12).SelectMany(_ => new int[]{3,4,5,6,7}).ToArray()),
  ("localite 60% page 0", Enumerable.Range(0,60).Select(_ => rngP.NextDouble() < 0.6 ? 0 : rngP.Next(1,8)).ToArray()),
  ("aleatoire uniforme", Enumerable.Range(0,60).Select(_ => rngP.Next(8)).ToArray())
};
var (noms, wfaA, lruA, fifoA, optA) = PagingTable(NPAGES, KCACHE, regimes, initP);
Console.WriteLine($"{"regime",24} {"WFA",5} {"LRU",5} {"FIFO",5} {"OPT",5} {"r_WFA",6} {"r_LRU",6}");
for (int i = 0; i < noms.Length; i++)
  Console.WriteLine($"{noms[i],24} {wfaA[i],5:0} {lruA[i],5:0} {fifoA[i],5:0} {optA[i],5:0} {wfaA[i] / optA[i],6:F2} {lruA[i] / optA[i],6:F2}");
display(new PlotSvg(SvgGroupedBars(noms, wfaA, lruA, fifoA, optA, optA[0] * 3,
  $"Paging : n={NPAGES} pages, cache k={KCACHE} — WFA et LRU sous le plafond k x OPT")));

                  regime   WFA   LRU  FIFO   OPT  r_WFA  r_LRU


  boucle 3-4-5-6-7 (x12)    46    60    60    32   1,44   1,88


     localite 60% page 0    21    26    28    17   1,24   1,53


      aleatoire uniforme    35    46    46    28   1,25   1,64


Paging : n=8 pages, cache k=3 — WFA et LRU sous le plafond k x OPT boucle 3-4-5-6-7 (x12) localite 60% page 0 aleatoire uniforme k x OPT (1re ligne) WFA LRU FIFO OPT (offline)

**Lecture** : sur la boucle, LRU faulte à chaque tour (60 faults) et son ratio franchit la
moitié du plafond sans briser la garantie (60 < k × OPT = 96), tandis que WFA (46 faults)
reste plus près de l'OPT. Sur la localité — le régime pour lequel LRU a été inventé —
l'écart se resserre sans s'inverser ; et sur l'aléatoire uniforme, WFA et LRU se rapprochent.
C'est tout le sens d'une garantie compétitive : **une borne sur le pire des cas**, achetée
au prix d'un moteur exact (énumératif) là où LRU est $O(1)$ par requête.

## 8. La work function, enfin vue

La preuve du preprint donne à la work function une représentation algébrique (matrices,
déterminants, potentiel) — hors de portée ici. Mais l'objet lui-même se **regarde** :
sur la ligne à n=4, k=2, il n'existe que 6 configurations ; la surface `wf_t(A)` au fil
des requêtes est une petite matrice que l'on peut afficher en heatmap. Deux choses à y
lire : les cellules **infinies** (configurations ne contenant pas la requête courante —
interdites à l'instant t), et la **re-multiplication des coûts** : après une requête en 3,
toutes les configurations viables contiennent 3.

In [9]:
// Heatmap SVG : grille wf_t(A) ; les cellules infinies (config interdite a l'instant t)
// sont gris clair. Couleurs bleu clair -> bleu fonce en fonction de la valeur.
string SvgHeatmap(double[,] grille, string[] colLabels, string[] rowLabels, string titre) {
  var sb = new StringBuilder();
  int rows = grille.GetLength(0), cols = grille.GetLength(1);
  int cw = 74, ch = 30, L = 92, T = 40;
  int W = L + cols * cw + 10, H = T + rows * ch + 22;
  double maxV = 0;
  for (int r = 0; r < rows; r++) for (int c = 0; c < cols; c++)
    if (grille[r, c] < double.PositiveInfinity && grille[r, c] > maxV) maxV = grille[r, c];
  if (maxV <= 0) maxV = 1;
  sb.Append($"<svg viewBox='0 0 {W} {H}' xmlns='http://www.w3.org/2000/svg' font-family='sans-serif'>");
  sb.Append($"<text x='{W / 2}' y='18' font-size='12' text-anchor='middle'>{titre}</text>");
  for (int r = 0; r < rows; r++) {
    for (int c = 0; c < cols; c++) {
      double v = grille[r, c];
      double x = L + c * cw, y = T + r * ch;
      string fill;
      if (v >= double.PositiveInfinity) fill = "#eeeeee";
      else {
        double t = v / maxV;
        int R = (int)(150 * (1 - t));
        int G = (int)(90 + 90 * (1 - t));
        int B = (int)(170 + 85 * t);
        fill = $"rgb({R},{G},{B})";
      }
      sb.Append($"<rect x='{x}' y='{y}' width='{cw}' height='{ch}' fill='{fill}' stroke='white'/>");
      string txtVal = v >= double.PositiveInfinity ? "inf" : v.ToString("0.#", CultureInfo.InvariantCulture);
      sb.Append($"<text x='{x + cw / 2}' y='{y + ch / 2 + 4}' font-size='10' text-anchor='middle'>{txtVal}</text>");
    }
    sb.Append($"<text x='{L - 6}' y='{T + r * ch + ch / 2 + 4}' font-size='9' text-anchor='end'>{rowLabels[r]}</text>");
  }
  for (int c = 0; c < cols; c++)
    sb.Append($"<text x='{L + c * cw + cw / 2}' y='{T + rows * ch + 14}' font-size='9' text-anchor='middle'>{colLabels[c]}</text>");
  sb.Append("</svg>");
  return sb.ToString();
}

int hn = 4, hk = 2;
var hdist = LigneMetric(hn);
var hreqs = new int[]{3,1,2,0,3};
var HC = AllConfigs(hn, hk);
var hinit = new int[]{0,1};

var hwf = new Dictionary<int[],double>();
foreach (var A in HC) hwf[A] = CfgDistance(hinit, A, hdist);
var colLabels = HC.Select(A => string.Join("", A)).ToArray();
var grille = new double[hreqs.Length + 1, HC.Count];
var hcur = (int[])hinit.Clone();
for (int c = 0; c < HC.Count; c++) grille[0, c] = hwf[HC[c]];
var trace = new List<int[]> { hcur };
for (int t = 0; t < hreqs.Length; t++) {
  int r = hreqs[t];
  var nw = new Dictionary<int[],double>();
  foreach (var A in HC) {
    if (!A.Contains(r)) nw[A] = double.PositiveInfinity;
    else {
      double best = double.PositiveInfinity;
      foreach (var B in HC) if (hwf[B] < double.PositiveInfinity) best = Math.Min(best, hwf[B] + CfgDistance(B, A, hdist));
      nw[A] = best;
    }
  }
  hwf = nw;
  hcur = HC.Where(A => A.Contains(r)).MinBy(A => hwf[A] + CfgDistance(hcur, A, hdist));
  for (int c = 0; c < HC.Count; c++) grille[t + 1, c] = hwf[HC[c]];
  trace.Add(hcur);
}
var rowLabels = new[] { "t=0 init" }
  .Concat(Enumerable.Range(0, hreqs.Length).Select(t => $"t={t + 1} (r={hreqs[t]})")).ToArray();
Console.WriteLine("Trace des configurations choisies par WFA : " + string.Join(" -> ", trace.Select(c => string.Join("", c))));
display(new PlotSvg(SvgHeatmap(grille, colLabels, rowLabels, "Work function sur la ligne n=4, k=2 — chaque ligne = un instant")));

Trace des configurations choisies par WFA : 01 -> 03 -> 13 -> 12 -> 02 -> 03


Work function sur la ligne n=4, k=2 — chaque ligne = un instant 0 1 2 2 3 4 t=0 init inf inf 2 inf 3 4 t=1 (r=3) 4 inf inf 4 3 inf t=2 (r=1) inf 5 inf 4 inf 4 t=3 (r=2) 6 5 6 inf inf inf t=4 (r=0) inf inf 6 inf 7 8 t=5 (r=3) 01 02 03 12 13 23

## 9. Ce que ce notebook a mesuré — et ce qu'il n'a pas prouvé

Le gabarit d'honnêteté d'une autre série ([RL-16](../../RL/RL-16-Dream-RSI-Python.ipynb) §8 —
« Ce que le jouet ne montre pas ») appliqué à la théorie :

| Affirmation | Statut | Preuve dans ce notebook |
|---|---|---|
| WFA ≤ k × OPT sur les instances testées | **mesuré** | §5 : 0 violation sur 600 instances seedées |
| ratio réalisé typique ~1.1-1.6 (max ≤ 2.0), loin de k | **mesuré** | §5-§6 : distributions, moyennes, max |
| LRU frôle la borne sans la briser (boucle k+2) | **mesuré** | §7 : 60 faults, k × OPT = 96 |
| WFA est k-compétitive sur **tout** espace métrique | **cité, non prouvé** | preprint arXiv 2609.15979 (14/09/2026, **non relu**) |
| l'écart (2k−1) → k tient | **cité, non prouvé** | idem — la technique (matrices, déterminants, potentiel) est hors scope |

Deux limites du banc à garder en tête :

1. **Le sampling ne rencontre pas le pire cas.** Les ratios max mesurés n'approchent pas k ;
   l'instance qui force le ratio k est construite adversarialement
   par la preuve, pas tirée au hasard. La garantie est une borne, pas une prédiction.
2. **Jouet = n ≤ 8, k ≤ 3.** Le moteur énumère les configurations (C(n,k)) et les
   appariements (k!) — exponentiel en k. C'est le prix de l'exactitude d'OPT ; les
   implémentations réelles de caching n'énumèrent rien (LRU/FIFO sont O(1) par requête),
   et la WFA exacte pour le paging est un objet d'étude, pas un produit.

## 10. Exercices

Trois exercices, du plus guidé au plus ouvert. Tous suivent la règle C.1 : le notebook
doit s'exécuter de bout en bout même non complété — les stubs ne lèvent jamais d'erreur.

### Exercice 1 — Double Coverage : l'algorithme dédié de la ligne

Sur la ligne, il existe un algorithme online **k-compétitif spécifique**, antérieur à la
conjecture : **Double Coverage** (Chrobak et al., 1991). À chaque requête r entre deux
serveurs voisins a < r < b, les deux serveurs avancent l'un vers l'autre à vitesse égale
(credit 1 partagé) jusqu'à ce que l'un atteigne r ; si r est à gauche de tous les serveurs
(ou à droite), seul le plus proche s'y déplace. Implémentez-le et comparez son coût à WFA
sur les mêmes suites — à parité de garantie, lequel paie le moins sur les instances jouets ?

*Indices : (1) représentez les serveurs comme une liste triée ; (2) le déplacement
symétrique peut être fait en une seule étape de coût min(r−a, b−r)×2... ou en pas
unitaires, les deux se valent pour le coût total ; (3) réutilisez `WfaRun` pour OPT.*

In [10]:
// Double Coverage sur la ligne (n points 0..n-1).
//   n : nombre de points ; requests : suite de requetes ; initServers : positions initiales.
// Retour : le cout total paye (float).
// Etape 1 : trier les serveurs et les garder tries.
// Etape 2 : pour chaque requete r, selon la position de r par rapport aux serveurs.
// Etape 3 : cas interieur -> les deux voisins avancent l'un vers l'autre.
// Etape 4 : cas exterieur -> seul le serveur le plus proche se deplace.
double? DoubleCoverage(int n, int[] requests, int[] initServers, Func<int,int,double> dist = null) {
  // TODO etudiant
  return null;
}

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 — Un ratio au-dessus de k n'est pas un contre-exemple

Dans le paging, LRU **est** k-compétitif (Sleator–Tarjan, 1985) : la garantie
s'énonce avec une **constante additive**, `faults(LRU) ≤ k·faults(OPT) + (k−1)`.
Le §7 a mesuré une forme plus étroite : **quand LRU et OPT partent du même cache
chaud**, le ratio *multiplicatif pur* `faults(LRU)/faults(OPT)` ne dépasse **jamais**
`k` — les deux caches partant du même état, la constante additive n'est pas nécessaire.
Le terme `(k−1)` n'agit que quand les deux caches **ne partent pas du même état** :
c'est le **transitoire de mise en charge**. LRU part d'un cache **vide** et paie, une
seule fois, le remplissage de ses `k` emplacements, pendant qu'OPT est analysé depuis
le cache **chaud** ; c'est dans ce cas froid/chaud que le ratio pur peut franchir `k`,
**sans réfuter la garantie** — la constante additive absorbe exactement ce transitoire.

Construisez une suite de requêtes sur n=8, k=3 où LRU **part d'un cache vide** et OPT
**du cache chaud**, de sorte que le ratio *pur* LRU/OPT dépasse k=3 ; puis vérifiez sur
cette même suite que l'inégalité à constante additive
`faults(LRU) ≤ k·faults(OPT) + (k−1)` **reste vraie**. C'est ce qui montre
qu'un ratio fini au-dessus de k n'est pas un contre-exemple.

*Indices : (1) l'asymétrie d'état de départ est la clé — `LruRun(n, k, reqs, [])`
pour LRU (cache vide) et `WfaRun(n, k, dist, reqs, init_pages)` pour OPT (cache
chaud `init_pages`) ; (2) `faults(OPT)` doit rester petit pour que le ratio pur dépasse
k — choisissez un cache chaud tel qu'OPT ne faulte presque plus ; (3) testez
`lru <= k*opt + (k-1)` sur la suite trouvée.*

In [11]:
// Construit une suite de requetes ou, LRU partant d'un cache VIDE et OPT du
// cache CHAUD, le ratio PUR LRU/OPT depasse k, tout en verifiant que l'inegalite
// a constante additive   faults(LRU) <= k*faults(OPT) + (k-1)
// reste vraie (un ratio fini > k n'est pas un contre-exemple : LRU est k-compétitif,
// la constante additive absorbe le transitoire de mise en charge).
// Mesure : LruRun(n, k, reqs, []) pour LRU (cache vide), WfaRun(n, k, dist, reqs,
// init) pour OPT (cache chaud `init` de taille k).
int[] SuiteAntiLru(int n = 8, int k = 3) {
  // TODO etudiant : retourner la liste des requetes.
  return null;
}

Console.WriteLine("Exercice a completer : une suite ou le ratio pur LRU/OPT > k (LRU cache vide, OPT cache chaud), puis verifier l'inegalite a constante additive.");

Exercice a completer : une suite ou le ratio pur LRU/OPT > k (LRU cache vide, OPT cache chaud), puis verifier l'inegalite a constante additive.


### Exercice 3 — Le prix de k : distribution des ratios sur cycle n=6

Toutes nos mesures ont utilisé k=2 ou k=3. Faites varier la **charge de serveurs** :
sur le cycle n=6, mesurez la distribution du ratio WFA/OPT pour k=2, k=3, k=4
(300 suites aléatoires chacun, seed fixé). Question ouverte à trancher par la mesure :
le ratio réalisé moyen **monte-t-il** avec k (chaque requête déplaçant potentiellement
plus de masse), ou reste-t-il plat ? Affichez les trois distributions superposées et
concluez en une phrase ce que cela suggère sur le « pire cas k » versus « moyenne ».

*Indices : (1) copiez le motif de la cellule §6 en paramétrant k ; (2) attention à
`initCfg` de taille k cohérente ; (3) superposez les trois histogrammes SVG avec des
couleurs distinctes.*

In [12]:
// Retourne {k: liste des ratios WFA/OPT} sur le cycle, par valeur de k.
Dictionary<int, List<double>> DistributionRatiosCycle(
    int n = 6, int[] ks = null, int nSuites = 300, int T = 30, int seed = 99) {
  // TODO etudiant
  return null;
}

Console.WriteLine("Exercice a completer");

Exercice a completer


## 11. Conclusion

- Le problème **k-server** formalise la décision online dans sa forme la plus pure :
  servir immédiatement, sans connaître la suite, payer la distance.
- La **work function** est à la fois l'OPT offline (sa récurrence de DP) et la mémoire
  qui pilote WFA ; un seul moteur a servi les deux rôles dans tout le notebook.
- Mesuré : ratios réalisés ~1.1-1.6 (maximum observé ≤ 2.0, garantie 2 ou 3 selon k) ; LRU
  au plus près de la borne sur la boucle (60 < 96) ; WFA bornée partout. Cité : la conjecture
  k-server (1990) tombée par le preprint arXiv 2609.15979 — la preuve elle-même (algèbre de
  matrices, potentiel) reste hors scope, et le papier n'est **pas encore relu**.
- **Correction de l'Exercice 2** : l'Exercice 2 montre qu'un ratio *pur* LRU/OPT ne dépasse
  k que quand LRU part d'un cache froid et OPT d'un cache chaud (le transitoire de mise en
  charge), et que la constante additive k−1 absorbe cet écart — un ratio fini au-dessus de k
  n'est donc pas un contre-exemple à la k-compétitivité de LRU, vraie dans le paging
  (Sleator-Tarjan, 1985).

**Pour aller plus loin** : la conjecture du secrétaire matroïdal est tombée 24 h plus tôt
(arXiv 2609.14555 — chaque élément de l'optimum accepté avec probabilité ≥ 1/4 en ne
connaissant que n et un oracle d'indépendance) ; la version randomisée de k-server
(borne inférieure k/(k−H_k) vs borne supérieure conjecturée k) ; le survey de Borodin et
El-Yaniv sur l'analyse compétitive.

Sœurs de la même fenêtre : `Complexity-04d` (l'algorithme de Singla
implémenté, garantie 1/4 comme banc de mesure) et
`Complexity/Complexity-04b` (le binôme synthétique — les deux
conjectures croisées, baseline 1/e et WFA, côté série Complexity).

## Références

- Coester, Koutsoupias, Zbysiński — *The k-server conjecture is true*, arXiv 2609.15979
  (soumis 14/09/2026, preprint non relu).
- Singla — *The Matroid Secretary Conjecture is True*, arXiv 2609.14555 (soumis 13/09/2026).
- Manasse, McGeoch, Sleator — *Competitive algorithms for server problems* (1990),
  la conjecture originelle.
- Koutsoupias, Papadimitriou — *On the k-server conjecture* (1995), WFA (2k−1)-compétitive.
- Chrobak, Karloff, Payne, Vishwanathan — *New results on server problems* (1991),
  Double Coverage sur la ligne.
- Sleator, Tarjan — *Amortized efficiency of list update and paging rules* (1985),
  la k-compétitivité de LRU dans le paging (constante additive).
- Borodin, El-Yaniv — *Online Computation and Competitive Analysis* (1998), le survey.

***
*Notebook Complexity 04c (C#) — distillation monde jouet d'un preprint de 48 h : la mesure honnête
d'une garantie, sans en démontrer la preuve. Jumeau C# de la version Python, et même correction
de l'Exercice 2. Le fil des décisions online se lit en amont : Complexity-04 (ski-rental,
pagination, secrétaire classique) puis Complexity-04b (les deux conjectures croisées).*

## À retenir

1. **La work function est un objet à deux lectures.** La même récurrence de programmation dynamique donne l'optimum offline ($\min_A \mathrm{wf}_T(A)$ = OPT) et la règle de décision online ($\mathrm{argmin}_A [\mathrm{wf}_t(A) + d(A_{t-1}, A)]$) — sur la suite tracée $[3, 1, 2, 0, 3]$ de la ligne à $n=4$, $k=2$, WFA paie 6 pour un OPT de 6 (ratio 1.00).
2. **La garantie pire-cas $k$ n'est jamais violée sur le banc.** Pour $k=3$, $n=6$, chacune des trois métriques (ligne, cycle, uniforme) rend « violations de $k=3$ : 0 » sur ses 200 instances seedées — un résultat **mesuré**, pas prouvé : le pire cas est une suite construite par la preuve, pas un tirage aléatoire.
3. **Le ratio réalisé vit loin du plafond.** Ligne $n=6$, $k=2$ et cycle $n=5$, $k=2$ : ratio moyen ~1.1-1.2, maximum ~1.4-1.5 — contre une garantie $k=2$. La borne est une ceinture de sécurité, pas une prédiction (les valeurs exactes sont dans les sorties des cellules §5-§6).
4. **Sur la boucle $k+2$, LRU frôle la borne sans la briser.** 60 faults pour LRU contre un OPT de 32 : le ratio 1.88 reste sous $k=3$ (60 < $k \times$ OPT = 96) ; WFA fait mieux (46 faults, ratio 1.44), au prix d'un moteur exact énumératif là où LRU est $O(1)$ par requête.
5. **Un ratio pur au-dessus de $k$ n'est pas un contre-exemple.** Quand LRU et OPT partent du même cache, le ratio LRU/OPT ne dépasse jamais $k$ (§7) ; il ne franchit $k$ que dans le cas froid/chaud de l'Exercice 2 — LRU d'un cache vide, OPT du cache chaud — où une courte suite donne p. ex. LRU $= 4$ pour OPT $= 1$ (ratio $4 > k = 3$) tout en satisfaisant $\mathrm{LRU} \le k\cdot\mathrm{OPT} + (k-1)$, soit $4 \le 3 \cdot 1 + 2 = 5$ : la constante additive absorbe le transitoire de mise en charge.
6. **L'adversaire naïf ne piège pas WFA.** L'oscillation $5/0$ répétée sur la ligne laisse WFA égaler l'OPT (4 contre 4, ratio 1.00) : elle place un serveur sur chacune puis ne bouge plus.

## Vérifiez votre compréhension

1. Pourquoi un seul moteur de programmation dynamique suffit-il à la fois pour calculer OPT et pour décider en ligne ?
2. Le banc de la section 5 rend « 0 violation » sur toutes ses instances seedées : pourquoi cela ne prouve-t-il pas le théorème ?
3. Sur la boucle 3-4-5-6-7, LRU paie 60 faults pour un OPT de 32 : pourquoi dit-on qu'il « frôle la borne sans la briser » — et dans quel cas le ratio pur LRU/OPT peut-il dépasser $k$ (et que construit l'Exercice 2) ?
4. Pourquoi l'oscillation entre les deux extrémités de la ligne donne-t-elle à WFA un ratio de 1.00 ?

<details><summary>Réponses</summary>

1. Parce que $\mathrm{wf}_t(A)$ est le coût minimal d'un trajet qui a servi toutes les requêtes et finit en $A$ : prendre le minimum sur $A$ à l'instant final donne l'optimum offline, tandis que la règle $\mathrm{argmin}$ sur $\mathrm{wf}_t(A) + d(A_{t-1}, A)$ réutilise la même table pour choisir la configuration suivante — une mémoire complète du passé au service du présent.
2. Parce que l'échantillonnage aléatoire ne rencontre pas la suite adversariale construite par la preuve ; les ratios mesurés restent typiquement entre 1.1 et 1.6, loin de $k$. « Jamais violée sur le banc » dit que l'implémentation est cohérente avec la garantie, pas que la garantie tient sur toute suite.
3. Parce que 60 reste sous $k \times$ OPT $= 3 \times 32 = 96$, avec un ratio LRU/OPT de 1.88 sous $k = 3$. Le ratio pur ne dépasse $k$ que lorsque LRU et OPT ne partent **pas** du même état — LRU d'un cache vide, OPT du cache chaud. L'Exercice 2 construit précisément une telle suite : $[3, 0, 1, 2]$ avec un cache chaud $\{3, 0, 1\}$ donne LRU $= 4$, OPT $= 1$, ratio $4 > k = 3$ — **tout en vérifiant que l'inégalité $\mathrm{LRU} \le k\cdot\mathrm{OPT} + (k-1)$ tient** ($4 \le 3 \cdot 1 + 2 = 5$) : ce ratio élevé n'est donc pas un contre-exemple à la k-compétitivité de LRU dans le paging, et WFA y reste sous le plafond.
4. Parce qu'après avoir servi chaque extrémité une fois, WFA place un serveur sur chacune : les requêtes suivantes tombent toujours sur un serveur déjà en place, et le coût s'arrête à 4, comme l'OPT. Le dur adversaire de WFA existe (la preuve le construit), mais il ne se rencontre pas par tirage au sort.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-04d — Le secrétaire matroïdal](Complexity-04d-Secretaire-Matroidal-Python.ipynb) — l'autre conjecture tombée à 24 h d'écart, mesurée élément par élément plutôt qu'instance par instance.
- **Jumeau** : [Complexity-04c — Version Python](Complexity-04c-KServer-WorkFunction-Python.ipynb) — même algorithme, métriques et budgets, et même correction de l'Exercice 2.
- **Dans la même série** : [Complexity-04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-CSharp.ipynb) — le socle licence : ski-rental, pagination, secrétaire classique ; [Complexity-04b — conjectures online](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) — le binôme synthétique qui croise les deux conjectures autour de la baseline 1/e.
- **Carnet voisin d'une autre série** : [RL-04 — Bandits manchots](../../RL/RL-04-Bandits-Manchots-Python.ipynb) — l'autre versant de la décision sous incertitude : explorer pour apprendre ce que le k-server suppose connu (la métrique), regret cumulé contre ratio compétitif.
- **Lectures** : Coester, Koutsoupias et Zbysiński, *The k-server conjecture is true* (arXiv 2609.15979, preprint non relu, cité en tête de ce carnet) ; Koutsoupias et Papadimitriou, *On the k-server conjecture* (1995), la garantie $(2k-1)$ qui a tenu trente et un ans ; Sleator et Tarjan, *Amortized efficiency of list update and paging rules* (1985), la k-compétitivité de LRU ; Borodin et El-Yaniv, *Online Computation and Competitive Analysis* (1998), le survey de référence.